[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/langchain_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估 LangChain 应用</h1>

LangChain 提供高级 API，几行代码即可构建强大的应用，但理解内部执行过程和定位问题往往并不容易。Phoenix 将查询引擎每次调用的底层结构可视化，并根据延迟、token 数及其他评估指标展示存在问题的 span，让 LLM 应用具备_可观测性_。

本教程将：

- 使用 LangChain 构建简单问答应用，通过检索增强生成回答有关 Arize 文档的问题
- 以 OpenInference 格式记录追踪数据
- 检查应用的追踪和 span，定位延迟和成本来源
- 将追踪导出为 pandas dataframe，使用 LLM 辅助评估衡量检索步骤的 precision@k

ℹ️ 本 Notebook 需要 OpenAI API 密钥。


## 1. 安装依赖并导入库

安装 Phoenix、LangChain 和 OpenAI。


In [ ]:
!pip install "arize-phoenix[evals]" langchain langchain-community langchain-openai openinference-instrumentation-langchain faiss-cpu

导入库。


In [ ]:
import json
import os
from getpass import getpass
from urllib.request import urlopen

import nest_asyncio
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from openinference.instrumentation.langchain import LangChainInstrumentor
from phoenix.client import Client
from phoenix.client.helpers.spans import get_input_output_context, get_retrieved_documents
from phoenix.evals import LLM, async_evaluate_dataframe, bind_evaluator
from phoenix.evals.metrics import (
    CorrectnessEvaluator,
    FaithfulnessEvaluator,
    RetrievalRelevanceEvaluator,
)
from phoenix.evals.utils import to_annotation_dataframe
from phoenix.otel import register
from tqdm import tqdm

import phoenix as px

nest_asyncio.apply()  # Notebook 环境并发评估所需

## 2. 启动 Phoenix

可以在后台运行 Phoenix，收集已通过 `OpenInferenceTracer` 添加埋点的 LangChain 应用发出的追踪数据。

启动 Phoenix，按照单元输出中的说明打开 Phoenix 界面。由于尚未运行 LangChain 应用，此时界面应为空。

In [ ]:
(session := px.launch_app()).view()

## 3. 配置 OpenAI API 密钥

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
if os.environ.get("OPENAI_API_KEY") is None:
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
    os.environ["OPENAI_API_KEY"] = openai_api_key

### 4. 构建 LangChain 应用

本示例在 Arize 文档上使用 LCEL 检索链，也可以换成自己的 LangChain 应用。

从云存储下载预构建的索引，并实例化存储上下文。


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
database_df = pd.read_parquet(
    "http://storage.googleapis.com/arize-phoenix-assets/datasets/unstructured/llm/context-retrieval/langchain-vertexai/database.parquet"
)
vectorstore = FAISS.from_texts(
    texts=database_df["text"].tolist(),
    embedding=embeddings,
)
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(model_name="gpt-3.5-turbo")

prompt = ChatPromptTemplate.from_template(
    "根据以下上下文回答问题：\n\n{context}\n\n问题：{question}"
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

为 LangChain 添加埋点


In [ ]:
tracer_provider = register()
LangChainInstrumentor(tracer_provider=tracer_provider).instrument(skip_dep_check=True)

## 5. 运行查询引擎并在 Phoenix 中查看追踪

下载一组关于 Arize 文档的常见查询样本。


In [ ]:
url = "http://storage.googleapis.com/arize-phoenix-assets/datasets/unstructured/llm/context-retrieval/arize_docs_queries.jsonl"
queries = []
with urlopen(url) as response:
    for line in response:
        line = line.decode("utf-8").strip()
        data = json.loads(line)
        queries.append(data["query"])
queries[:10]

运行几个查询。


In [ ]:
for query in tqdm(queries[:10]):
    chain.invoke(query)

查询运行时查看 Phoenix 界面，追踪应实时出现。


## 6. 导出与评估追踪数据

可以将追踪导出为 pandas dataframe，以便进一步分析和评估。

本例会将 `retriever` span 导出为两个独立的 dataframe：

- `queries_df`：将每个查询检索到的文档拼接到一列中
- `retrieved_documents_df`：将每篇检索文档展开为独立的一行，以便单独评估每个查询与文档的配对

这支持计算多种评估指标：

- 相关性：检索到的文档是否为回答提供依据？
- 问答正确性：应用的回答是否基于检索到的上下文？
- 幻觉：应用是否编造了虚假信息？


In [ ]:
queries_df = get_input_output_context(Client())
retrieved_documents_df = get_retrieved_documents(Client())

接下来定义评估模型和评估器。

评估器基于语言模型，通过提示 LLM 判断回答质量、检索文档相关性等，即使没有人工标注数据，也能提供质量信号。选择评估器类型，再使用目标语言模型实例化，即可利用经过实践验证的评估模板。

注意：`FaithfulnessEvaluator` 衡量输出对上下文的忠实程度，分数越高，表示输出越有依据、越少幻觉。

In [ ]:
eval_model = LLM(
    provider="openai",
    model="gpt-4-turbo-preview",
)
faithfulness_evaluator = FaithfulnessEvaluator(llm=eval_model)
correctness_evaluator = CorrectnessEvaluator(llm=eval_model)

# 将 get_retrieved_documents 的 "document" 列映射到评估器的 "context" 字段
relevance_evaluator = bind_evaluator(
    evaluator=RetrievalRelevanceEvaluator(llm=eval_model),
    input_mapping={"context": "document"},
)

eval_results = await async_evaluate_dataframe(
    dataframe=queries_df,
    evaluators=[faithfulness_evaluator, correctness_evaluator],
    concurrency=10,
)
relevance_results = await async_evaluate_dataframe(
    dataframe=retrieved_documents_df,
    evaluators=[relevance_evaluator],
    concurrency=10,
)

annotations = to_annotation_dataframe(eval_results)
relevance_annotations = to_annotation_dataframe(relevance_results)

from phoenix.client import AsyncClient

px_client = AsyncClient()
await px_client.spans.log_span_annotations_dataframe(
    dataframe=annotations,
)

Client().spans.log_document_annotations_dataframe(
    dataframe=relevance_annotations,
    annotation_name="Relevance",
    annotator_kind="LLM",
)

评估结果现在应以标注的形式显示在 Phoenix 中对应的 span 上。


In [ ]:
print(f"🚀 若尚未打开，请打开 Phoenix 界面： {session.url}")

## 7. 总结

LLM 追踪及其配套的 OpenInference Tracing 规范用于描述一类遥测数据，帮助理解 LLM 执行及周边应用上下文，例如向量库检索、搜索引擎或 API 等外部工具的使用。它既揭示应用各步骤的内部执行过程，也展示整个系统的运行表现。

LLM Evals 提供简单、快速、准确的 LLM 评估，帮助快速衡量 LLM 应用表现并定位执行中存在问题的 span。

更多信息请参阅[文档](https://arize.com/docs/phoenix/)。
